<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/GNN_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Run these first in Colab
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install cuml-cu12  # This is the big one for KNN

Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 109.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 69.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


In [8]:
# -*- coding: utf-8 -*-
"""GNN v2 Enhanced — Irrigation Need
Improvements over v1:
  1. Decision Stump Thresholding (binary feature flags per numeric feature)
  2. Rank Percentile Encoding (replaces StandardScaler for numeric features)
  3. Digit-Level Features (first decimal digit as categorical signal)
  4. Autoencoder Bottleneck Features (4-dim latent representation)
  5. Custom MultiBalancedAccuracy metric + EarlyStopping on val_balanced_acc
"""

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Run these first in Colab
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install cuml-cu12

import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.tree import DecisionTreeClassifier

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

try:
    from cuml.neighbors import NearestNeighbors
    print("🚀 Using RAPIDS cuML KNN - Very Fast!")
except ImportError:
    from sklearn.neighbors import NearestNeighbors
    print("⚠️ cuML not found → Falling back to sklearn KNN (slower)")

# ============================================================
# CONFIG
# ============================================================
OUT_DIR    = "/content/drive/MyDrive/irrigation_need_v15/"
TRAIN_PATH = "/content/drive/MyDrive/irrigation_need_v15/train.csv"
TEST_PATH  = "/content/drive/MyDrive/irrigation_need_v15/test.csv"
VERSION_NB = "GNN_v2_enhanced"
SEED       = 42
N_FOLDS    = 5
K          = 8
EPOCHS     = 80
PATIENCE   = 15
BATCH_SIZE = 4096
INFER_BATCH = 8192
FANOUTS    = [6, 4]
GRAPH_NUM_MULTIPLIER = 3.0
USE_AMP    = True
RARE_MIN   = 25
HIDDEN     = 128
DROPOUT    = 0.20
LR         = 8e-4
WEIGHT_DECAY = 3e-4

# Autoencoder config
AE_LATENT_DIM  = 4
AE_EPOCHS      = 30
AE_BATCH_SIZE  = 2048
AE_LR          = 1e-3

TARGET    = "Irrigation_Need"
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2}
LABEL_INV = {0: "Low", 1: "Medium", 2: "High"}
N_CLASSES = 3

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"
]

CAT_PROXY  = [f"{c}__cat"     for c in NUMS]
CAT_RARE   = [f"{c}__is_rare" for c in NUMS]
# Digit features and stump features are added dynamically in engineer_features
STUMP_COLS  = [f"{c}__stump" for c in NUMS]   # binary flags
DIGIT_COLS  = [f"{c}__digit" for c in NUMS]   # decimal digit 0–9
AE_COLS     = [f"ae_feat_{i}" for i in range(AE_LATENT_DIM)]

ALL_CATS = CATS + CAT_PROXY + CAT_RARE + STUMP_COLS + DIGIT_COLS
ALL_NUMS = NUMS[:] + AE_COLS  # raw numerics + AE latent features

GRAPH_CAT_COLS = CATS[:]
GRAPH_NUM_COLS = NUMS[:]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")

N_GPUS = torch.cuda.device_count()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# IMPROVEMENT 1 — DECISION STUMP THRESHOLDING
# ============================================================

def build_decision_stumps(train_df: pd.DataFrame, y: np.ndarray):
    """
    Fit a DecisionTreeClassifier(max_depth=1) per numeric column.
    Returns a dict mapping col → threshold value (float).
    These thresholds are fit on train only and applied to train+test.

    Why: creates a hard binary signal for each feature at the point
    where a stump would naturally cut the data. Linear models in the
    meta-stacker (e.g. Ridge) cannot learn non-linear thresholds by
    themselves; these flags hand them that information directly.
    """
    stumps = {}
    for col in NUMS:
        x = train_df[col].values.reshape(-1, 1).astype(np.float32)
        dt = DecisionTreeClassifier(max_depth=1, random_state=SEED)
        dt.fit(x, y)
        # Extract the single split threshold from the tree
        threshold = dt.tree_.threshold[0]   # root node threshold
        stumps[col] = float(threshold)
    return stumps


def apply_decision_stumps(df: pd.DataFrame, stumps: dict) -> pd.DataFrame:
    """
    Add binary __stump columns: 1 if value >= threshold, 0 otherwise.
    Fit stumps on train only; apply to both train and test.
    """
    df = df.copy()
    for col, thr in stumps.items():
        df[f"{col}__stump"] = (df[col].astype(np.float32) >= thr).astype(np.int8)
    return df


# ============================================================
# IMPROVEMENT 2 — RANK PERCENTILE ENCODING
# ============================================================

class RankPercentileEncoder:
    """
    Converts each numeric column into its rank percentile in [0, 1].
    Fit on train, applied to train+test using interpolation for unseen values.

    Why: rank-based scaling is immune to outliers and 'stretches' dense
    clusters, making it easier for the GNN to find decision boundaries in
    crowded feature regions. Equivalent to GaussRank without the final
    Gaussian step.
    """
    def __init__(self):
        self.sorted_values_ = {}   # col → sorted train array

    def fit(self, df: pd.DataFrame, cols: list):
        for col in cols:
            vals = df[col].values.astype(np.float32)
            self.sorted_values_[col] = np.sort(vals[~np.isnan(vals)])
        return self

    def transform(self, df: pd.DataFrame, cols: list) -> np.ndarray:
        out = np.zeros((len(df), len(cols)), dtype=np.float32)
        for j, col in enumerate(cols):
            vals = df[col].values.astype(np.float32)
            sv   = self.sorted_values_[col]
            n    = len(sv)
            # searchsorted gives position → divide by n for percentile in (0,1]
            ranks = np.searchsorted(sv, vals, side="left").astype(np.float32)
            out[:, j] = ranks / max(n, 1)
        return out

    def fit_transform(self, df: pd.DataFrame, cols: list) -> np.ndarray:
        self.fit(df, cols)
        return self.transform(df, cols)


# ============================================================
# IMPROVEMENT 3 — DIGIT-LEVEL FEATURES
# ============================================================

def add_digit_features(df: pd.DataFrame, cols: list, k: int = 1) -> pd.DataFrame:
    """
    Extract the k-th decimal digit from each numeric column.

    Formula: floor(value * 10^k + 1e-9) % 10 → digit in {0..9}

    Why: in sensor datasets, the decimal precision (e.g. .99 vs .01)
    often reflects a sensor's quantization state or rounding behavior
    that correlates with the target. Treating the digit as a categorical
    feature surfaces this signal cheaply.

    k=1 → first decimal digit (e.g. 25.7 → 7)
    """
    df = df.copy()
    for col in cols:
        vals = pd.to_numeric(df[col], errors="coerce").astype(np.float64)
        digit = (np.floor(vals * (10 ** k) + 1e-9) % 10).astype(np.int8)
        df[f"{col}__digit"] = digit.astype(str)   # treat as categorical
    return df


# ============================================================
# IMPROVEMENT 4 — AUTOENCODER BOTTLENECK FEATURES
# ============================================================

class _Autoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train_autoencoder(
    Xn_train: np.ndarray,
    Xn_test:  np.ndarray,
    latent_dim: int = AE_LATENT_DIM,
    epochs:     int = AE_EPOCHS,
    batch_size: int = AE_BATCH_SIZE,
    lr:         float = AE_LR,
    device: torch.device = DEVICE,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Train an unsupervised Autoencoder on all (train+test) numeric features.
    Returns latent representations for train and test.

    Why: the bottleneck layer learns a compressed 'summary' of the
    soil/weather conditions. Adding ae_feat columns gives the stacker
    a holistic environment descriptor not captured by individual features.
    """
    print(f"\n[Autoencoder] Training AE  latent_dim={latent_dim}  epochs={epochs}")
    Xn_all = np.vstack([Xn_train, Xn_test]).astype(np.float32)
    X_tensor = torch.tensor(Xn_all, dtype=torch.float32)

    model = _Autoencoder(input_dim=Xn_all.shape[1], latent_dim=latent_dim).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    ds    = torch.utils.data.TensorDataset(X_tensor)
    dl    = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for (batch_x,) in dl:
            batch_x = batch_x.to(device)
            recon, _ = model(batch_x)
            loss = F.mse_loss(recon, batch_x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(batch_x)
        if epoch % 5 == 0 or epoch == 1:
            print(f"  AE Epoch {epoch:03d} | MSE: {total_loss / len(Xn_all):.6f}")

    # Extract latent features for all rows
    model.eval()
    with torch.no_grad():
        _, z_all = model(X_tensor.to(device))
    z_all = z_all.cpu().numpy().astype(np.float32)
    del model, opt, ds, dl, X_tensor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    n_train = Xn_train.shape[0]
    print(f"[Autoencoder] Done. Latent features shape (all): {z_all.shape}")
    return z_all[:n_train], z_all[n_train:]


# ============================================================
# IMPROVEMENT 5 — MULTI-CLASS BALANCED ACCURACY (PyTorch)
# ============================================================

class BalancedAccuracyTracker:
    """
    Tracks balanced accuracy during GNN training using PyTorch tensors.
    Compatible with the existing training loop (no Keras dependency).

    Usage: call .update(logits, labels) each batch, .compute() at epoch end.

    Why: standard accuracy ignores class imbalance. BA weights each class
    equally, mirroring the competition metric. Using val_ba for early
    stopping prevents the model from terminating simply because it has
    learned the dominant 'Low' class well.
    """
    def __init__(self, n_classes: int = N_CLASSES):
        self.n_classes = n_classes
        self.reset()

    def reset(self):
        self.confusion = np.zeros((self.n_classes, self.n_classes), dtype=np.int64)

    def update(self, probs: np.ndarray, labels: np.ndarray):
        preds = np.argmax(probs, axis=1)
        for t, p in zip(labels, preds):
            self.confusion[int(t), int(p)] += 1

    def compute(self) -> float:
        per_class_recall = []
        for i in range(self.n_classes):
            row_sum = self.confusion[i].sum()
            recall  = self.confusion[i, i] / row_sum if row_sum > 0 else 0.0
            per_class_recall.append(recall)
        return float(np.mean(per_class_recall))


# ============================================================
# GENETIC THRESHOLD / BIAS HELPERS  (unchanged from v1)
# ============================================================

def apply_thresholds(probs, thresholds):
    adjusted = probs - thresholds
    return np.argmax(adjusted, axis=1)

def fitness_thresholds(thresholds, probs, y):
    preds = apply_thresholds(probs, thresholds)
    return -balanced_accuracy_score(y, preds)

def optimize_thresholds(probs, y, pop_size=30, generations=40, mutation_scale=0.05):
    n_classes  = probs.shape[1]
    population = [np.random.uniform(-0.2, 0.2, size=n_classes) for _ in range(pop_size)]
    for gen in range(generations):
        scores = np.array([fitness_thresholds(ind, probs, y) for ind in population])
        idx    = np.argsort(scores)
        population = [population[i] for i in idx[:pop_size // 2]]
        children   = []
        while len(children) < pop_size // 2:
            p1, p2 = random.sample(population, 2)
            child  = (p1 + p2) / 2 + np.random.normal(0, mutation_scale, size=n_classes)
            children.append(child)
        population.extend(children)
        print(f"Gen {gen:02d} | Best BA: {-scores[idx[0]]:.5f}")
    scores = np.array([fitness_thresholds(ind, probs, y) for ind in population])
    return population[np.argmin(scores)]

def apply_bias(probs, bias):
    logits = np.log(np.clip(probs, 1e-9, 1.0)) + bias
    exp    = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def fitness(bias, probs, y_true):
    return log_loss(y_true, apply_bias(probs, bias), labels=[0, 1, 2])

def genetic_optimize(probs, y, pop_size=30, generations=40, mutation_scale=0.1):
    n_classes  = probs.shape[1]
    population = [np.random.uniform(-0.5, 0.5, size=n_classes) for _ in range(pop_size)]
    for gen in range(generations):
        scores = np.array([fitness(ind, probs, y) for ind in population])
        idx    = np.argsort(scores)
        population = [population[i] for i in idx[:pop_size // 2]]
        children   = []
        while len(children) < pop_size // 2:
            p1, p2 = random.sample(population, 2)
            child  = (p1 + p2) / 2 + np.random.normal(0, mutation_scale, size=n_classes)
            children.append(child)
        population.extend(children)
        print(f"Gen {gen:02d} | Best LogLoss: {scores[idx[0]]:.5f}")
    scores = np.array([fitness(ind, probs, y) for ind in population])
    return population[np.argmin(scores)]


# ============================================================
# PREPROCESSING  (unchanged from v1)
# ============================================================

def preprocess(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = train_df.copy()
    te = test_df.copy()
    for c in NUMS:
        tr[c] = pd.to_numeric(tr[c], errors="coerce").astype(np.float32)
        te[c] = pd.to_numeric(te[c], errors="coerce").astype(np.float32)
        med = float(np.nanmedian(tr[c].values))
        tr[c] = tr[c].fillna(med)
        te[c] = te[c].fillna(med)
    for c in CATS:
        tr[c] = tr[c].astype(str).str.strip().fillna("missing")
        te[c] = te[c].astype(str).str.strip().fillna("missing")
    y = tr[TARGET].values.astype(np.int64)
    print("Unique labels:", np.unique(y))
    assert not np.isnan(y).any(), "NaNs in labels!"
    assert set(np.unique(y)) <= {0, 1, 2}, "Invalid labels detected!"
    return tr, te, y


# ============================================================
# FEATURE ENGINEERING  (extended with improvements 1, 2, 3)
# ============================================================

def _build_snapper(train_series: pd.Series):
    s  = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = s.value_counts(dropna=False)
    frequent = np.sort(
        np.array([v for v in vc[vc >= RARE_MIN].index if pd.notna(v)], dtype=np.float32)
    )
    if frequent.size == 0:
        frequent = np.sort(s.dropna().unique().astype(np.float32))
    freq_set = set(frequent.tolist())

    def transform(series):
        x       = pd.to_numeric(series, errors="coerce").astype(np.float32).values
        is_nan  = np.isnan(x)
        is_rare = np.ones(len(x), dtype=np.int32)
        for i, v in enumerate(x):
            if not np.isnan(v) and float(v) in freq_set:
                is_rare[i] = 0
        x_snapped = x.copy()
        snap_idx  = np.where((~is_nan) & (is_rare == 1))[0]
        if snap_idx.size > 0 and frequent.size > 0:
            v      = x[snap_idx]
            pos    = np.clip(np.searchsorted(frequent, v), 0, len(frequent) - 1)
            left   = np.clip(pos - 1, 0, len(frequent) - 1)
            right  = pos
            nearest = np.where(
                np.abs(v - frequent[right]) <= np.abs(v - frequent[left]),
                frequent[right], frequent[left]
            )
            x_snapped[snap_idx] = nearest.astype(np.float32)
        return x_snapped.astype(np.float32), is_rare.astype(np.int32)

    return transform


def engineer_features(
    train_df: pd.DataFrame,
    test_df:  pd.DataFrame,
    y_train:  np.ndarray,
):
    """
    Builds all node features:
      - Rare-snap __cat / __is_rare flags (v1)
      - Decision stump binary flags  __stump  (improvement 1)
      - Digit-level decimal features __digit  (improvement 3)
    Returns enriched train/test DataFrames.
    """
    print("\n[Feature Engineering] Building features...")
    tr = train_df.copy()
    te = test_df.copy()

    # --- Rare-snap features (v1, unchanged) ---
    for col in NUMS:
        snapper = _build_snapper(tr[col])
        tr_snap, tr_rare = snapper(tr[col])
        te_snap, te_rare = snapper(te[col])
        tr[f"{col}__cat"]     = pd.Series(tr_snap).astype(str).values
        te[f"{col}__cat"]     = pd.Series(te_snap).astype(str).values
        tr[f"{col}__is_rare"] = pd.Series(tr_rare).astype(str).values
        te[f"{col}__is_rare"] = pd.Series(te_rare).astype(str).values

    # --- IMPROVEMENT 1: Decision Stump Thresholding ---
    print("[Feature Engineering] Fitting decision stumps...")
    stumps = build_decision_stumps(tr, y_train)
    tr = apply_decision_stumps(tr, stumps)
    te = apply_decision_stumps(te, stumps)
    # Convert stump binary flags to string for categorical encoding
    for col in NUMS:
        tr[f"{col}__stump"] = tr[f"{col}__stump"].astype(str)
        te[f"{col}__stump"] = te[f"{col}__stump"].astype(str)
    print(f"  Stump thresholds: { {k: round(v,4) for k,v in stumps.items()} }")

    # --- IMPROVEMENT 3: Digit-Level Features ---
    print("[Feature Engineering] Extracting digit-level features...")
    tr = add_digit_features(tr, NUMS, k=1)
    te = add_digit_features(te, NUMS, k=1)

    # Ensure all categorical columns are strings
    for df in (tr, te):
        for c in ALL_CATS:
            if c in df.columns:
                df[c] = df[c].astype(str).fillna("missing")

    print(f"[Feature Engineering] Categorical features: {len(ALL_CATS)}")
    print(f"[Feature Engineering] Numeric features    : {len(NUMS)} raw + {AE_LATENT_DIM} AE latent")
    return tr, te, stumps


def encode_categoricals(train_df: pd.DataFrame, test_df: pd.DataFrame):
    print("\n[Encode] Encoding categorical node features...")
    tr_codes, te_codes, cardinalities = [], [], []
    for c in ALL_CATS:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        all_vals = pd.concat(
            [train_df[c].astype(str), test_df[c].astype(str)], ignore_index=True
        )
        mapping = {v: i for i, v in enumerate(all_vals.unique())}
        tr_codes.append(train_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        te_codes.append(test_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        cardinalities.append(len(mapping))
    Xc_tr = np.stack(tr_codes, axis=1)
    Xc_te = np.stack(te_codes, axis=1)
    print(f"[Encode] Categorical matrix — train: {Xc_tr.shape} | test: {Xc_te.shape}")
    return Xc_tr, Xc_te, cardinalities


def scale_numerics(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    IMPROVEMENT 2 — Rank Percentile Encoding replaces StandardScaler.
    Fit on train only; applied to train and test via interpolation.
    Returns arrays of shape [n, len(NUMS)] with values in [0, 1].
    AE latent features are appended in the main pipeline after AE training.
    """
    print("\n[Scale] Rank-percentile encoding numeric node features...")
    rpe = RankPercentileEncoder()
    Xn_tr = rpe.fit_transform(train_df, NUMS)
    Xn_te = rpe.transform(test_df, NUMS)
    print(f"[Scale] Numeric matrix — train: {Xn_tr.shape} | test: {Xn_te.shape}")
    return Xn_tr, Xn_te, rpe


def build_knn_graph(train_df: pd.DataFrame, test_df: pd.DataFrame, k: int = K):
    print(f"\n[Graph] Building KNN graph (k={k}) on {len(train_df) + len(test_df):,} nodes...")
    graph_cat = pd.concat(
        [train_df[GRAPH_CAT_COLS].astype(str), test_df[GRAPH_CAT_COLS].astype(str)],
        ignore_index=True
    )
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
    X_cat_ohe = ohe.fit_transform(graph_cat).astype(np.float32)

    graph_num_tr = train_df[GRAPH_NUM_COLS].copy()
    graph_num_te = test_df[GRAPH_NUM_COLS].copy()
    for c in GRAPH_NUM_COLS:
        graph_num_tr[c] = pd.to_numeric(graph_num_tr[c], errors="coerce").fillna(0).astype(np.float32)
        graph_num_te[c] = pd.to_numeric(graph_num_te[c], errors="coerce").fillna(0).astype(np.float32)

    # Use RankPercentile for graph features too (consistent with node features)
    rpe_graph = RankPercentileEncoder()
    X_num_tr  = rpe_graph.fit_transform(graph_num_tr, GRAPH_NUM_COLS)
    X_num_te  = rpe_graph.transform(graph_num_te, GRAPH_NUM_COLS)
    X_num     = np.vstack([X_num_tr, X_num_te]).astype(np.float32) * GRAPH_NUM_MULTIPLIER
    X_graph   = np.concatenate([X_cat_ohe, X_num], axis=1).astype(np.float32)
    print(f"[Graph] Graph feature matrix shape: {X_graph.shape}")

    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(X_graph)
    _, idx = knn.kneighbors(X_graph)
    if hasattr(idx, "get"):
        idx = idx.get()
    neighbors = idx.astype(np.int32)
    print(f"[Graph] Neighbors matrix shape: {neighbors.shape}")
    del X_graph, X_cat_ohe, X_num, idx, knn
    gc.collect()
    return neighbors


# ============================================================
# MODEL DEFINITION (unchanged from v1, num_in auto-updated)
# ============================================================

def _emb_dim(cardinality: int) -> int:
    return int(np.clip(round(1.8 * (cardinality ** 0.25)), 4, 24))


class CatEmbed(nn.Module):
    def __init__(self, cardinalities):
        super().__init__()
        self.embs    = nn.ModuleList()
        self.out_dim = 0
        for card in cardinalities:
            card = max(2, int(card))
            d    = _emb_dim(card)
            self.embs.append(nn.Embedding(card, d))
            self.out_dim += d
        for e in self.embs:
            nn.init.normal_(e.weight, 0.0, 0.02)

    def forward(self, x_cat):
        return torch.cat([emb(x_cat[:, j]) for j, emb in enumerate(self.embs)], dim=1)


class IrrigationGNN(nn.Module):
    def __init__(self, num_in: int, cardinalities: list, hidden: int = 128, dropout: float = 0.2):
        super().__init__()
        self.cat    = CatEmbed(cardinalities)
        in_dim      = num_in + self.cat.out_dim
        self.lin_in = nn.Linear(in_dim, hidden)
        self.conv1  = SAGEConv(hidden, hidden)
        self.conv2  = SAGEConv(hidden, hidden)
        self.norm1  = nn.LayerNorm(hidden)
        self.norm2  = nn.LayerNorm(hidden)
        self.drop   = dropout
        self.head   = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, N_CLASSES),
        )

    def forward(self, data):
        x  = torch.cat([data.x_num, self.cat(data.x_cat)], dim=1)
        x  = F.dropout(F.relu(self.lin_in(x)), p=self.drop, training=self.training)
        x1 = F.relu(self.norm1(self.conv1(x,  data.edge_index)))
        x1 = F.dropout(x1, p=self.drop, training=self.training)
        x  = x + 0.5 * x1
        x2 = F.relu(self.norm2(self.conv2(x, data.edge_index)))
        x2 = F.dropout(x2, p=self.drop, training=self.training)
        x  = x + 0.5 * x2
        return self.head(x)


# ============================================================
# SUBGRAPH UTILITIES (unchanged)
# ============================================================

_global_pos = None


def _build_subgraph(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu, fanouts, device, offset=0):
    global _global_pos
    seed_nodes = np.asarray(seed_nodes, dtype=np.int32)
    frontier   = seed_nodes
    collected  = [seed_nodes]
    for hop, fanout in enumerate(fanouts):
        nbr   = neighbors[frontier]
        start = (offset + hop) % nbr.shape[1]
        cols  = (np.arange(fanout) + start) % nbr.shape[1]
        frontier = np.unique(nbr[:, cols].reshape(-1))
        collected.append(frontier)
    nodes = np.unique(np.concatenate(collected))
    m     = len(nodes)
    _global_pos[nodes] = np.arange(m, dtype=np.int32)
    sub_nbr    = neighbors[nodes]
    dst_local  = _global_pos[sub_nbr]
    mask       = dst_local >= 0
    src_l      = np.repeat(np.arange(m, dtype=np.int64), sub_nbr.shape[1])[mask.reshape(-1)]
    dst_l      = dst_local[mask].astype(np.int64)
    edge_index = torch.tensor(np.vstack([src_l, dst_l]), dtype=torch.long, device=device)
    batch = Data(
        x_num      = x_num_cpu[nodes].to(device),
        x_cat      = x_cat_cpu[nodes].to(device, non_blocking=True),
        y          = y_cpu[nodes].to(device, non_blocking=True),
        edge_index = edge_index,
    )
    batch.seed_local = torch.tensor(_global_pos[seed_nodes], dtype=torch.long, device=device)
    _global_pos[nodes] = -1
    return batch


def _seed_batches(seed_nodes, batch_size, shuffle):
    arr = np.asarray(seed_nodes, dtype=np.int32).copy()
    if shuffle:
        np.random.shuffle(arr)
    for i in range(0, len(arr), batch_size):
        yield arr[i:i + batch_size]


@torch.no_grad()
def _predict_nodes(model, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu,
                   fanouts, batch_size, device, offset=0):
    model.eval()
    out = np.zeros((len(seed_nodes), N_CLASSES), dtype=np.float32)
    pos = 0
    for batch_seeds in _seed_batches(seed_nodes, batch_size, shuffle=False):
        batch  = _build_subgraph(batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                                  y_cpu, fanouts, device, offset)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(USE_AMP and device.type == "cuda")):
            logits = model(batch)
        probs = F.softmax(logits[batch.seed_local], dim=1).float().cpu().numpy()
        out[pos:pos + len(batch_seeds)] = probs
        pos += len(batch_seeds)
        del batch, logits, probs
    return out


# ============================================================
# SKLEARN WRAPPER  (improved with BalancedAccuracyTracker)
# ============================================================

class IrrigationGNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        hidden       = HIDDEN,
        dropout      = DROPOUT,
        lr           = LR,
        weight_decay = WEIGHT_DECAY,
        epochs       = EPOCHS,
        patience     = PATIENCE,
        batch_size   = BATCH_SIZE,
        infer_batch  = INFER_BATCH,
        fanouts      = FANOUTS,
        device       = DEVICE,
        use_amp      = USE_AMP,
        n_gpus       = None,
        # IMPROVEMENT 5: early stopping target; 'val_loss' or 'val_ba'
        early_stop_on = "val_ba",
    ):
        self.hidden        = hidden
        self.dropout       = dropout
        self.lr            = lr
        self.weight_decay  = weight_decay
        self.epochs        = epochs
        self.patience      = patience
        self.batch_size    = batch_size
        self.infer_batch   = infer_batch
        self.fanouts       = fanouts
        self.device        = device
        self.use_amp       = use_amp
        self.early_stop_on = early_stop_on
        self.model_        = None
        self.cardinalities_= None
        self.n_gpus        = n_gpus if n_gpus is not None else torch.cuda.device_count()

    def fit(
        self,
        train_idx,
        val_idx,
        y_all,
        neighbors,
        x_num_cpu,
        x_cat_cpu,
        y_cpu,
        cardinalities,
        class_weights,
    ):
        global _global_pos
        n_all       = x_num_cpu.shape[0]
        _global_pos = np.full(n_all, -1, dtype=np.int32)
        self.cardinalities_ = cardinalities

        core_model = IrrigationGNN(
            num_in       = x_num_cpu.shape[1],
            cardinalities= cardinalities,
            hidden       = self.hidden,
            dropout      = self.dropout,
        )
        if self.n_gpus > 1:
            core_model = nn.DataParallel(core_model)
        self.model_ = core_model.to(self.device)

        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(self.device))
        opt     = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scaler  = torch.cuda.amp.GradScaler(enabled=(self.use_amp and self.device.type == "cuda"))

        # IMPROVEMENT 5: track BA per epoch alongside loss
        ba_tracker = BalancedAccuracyTracker(n_classes=N_CLASSES)

        # For 'val_ba' we maximise; for 'val_loss' we minimise
        best_val_metric = -float("inf") if self.early_stop_on == "val_ba" else float("inf")
        best_state      = None
        bad_epochs      = 0

        print(f"    Model parameters : {sum(p.numel() for p in self.model_.parameters()):,}")
        print(f"    Training nodes   : {len(train_idx):,}  |  Validation nodes: {len(val_idx):,}")
        print(f"    Early stopping on: {self.early_stop_on}  |  Patience: {self.patience}\n")

        for epoch in range(1, self.epochs + 1):
            # ---- train ----
            self.model_.train()
            epoch_losses = []
            offset = epoch % K

            for batch_seeds in _seed_batches(train_idx, self.batch_size, shuffle=True):
                batch = _build_subgraph(
                    batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                    y_cpu, self.fanouts, self.device, offset
                )
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16,
                                    enabled=(self.use_amp and self.device.type == "cuda")):
                    logits = self.model_(batch)
                    loss   = loss_fn(logits[batch.seed_local], batch.y[batch.seed_local].long())
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(self.model_.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                epoch_losses.append(loss.item())
                del batch, logits, loss

            # ---- validate ----
            val_probs    = _predict_nodes(
                self.model_, val_idx, neighbors, x_num_cpu, x_cat_cpu,
                y_cpu, self.fanouts, self.infer_batch, self.device, offset
            )
            y_val_true   = y_all[val_idx]
            val_loss     = log_loss(y_val_true, val_probs, labels=[0, 1, 2])

            # IMPROVEMENT 5: compute val BA via tracker
            ba_tracker.reset()
            ba_tracker.update(val_probs, y_val_true)
            val_ba = ba_tracker.compute()

            print(
                f"    Epoch {epoch:04d} | "
                f"Train Loss: {np.mean(epoch_losses):.5f} | "
                f"Val LogLoss: {val_loss:.5f} | "
                f"Val BA: {val_ba:.5f}"
            )

            # ---- early stopping (on val_ba or val_loss) ----
            if self.early_stop_on == "val_ba":
                improved = val_ba > best_val_metric + 1e-6
                current  = val_ba
            else:
                improved = val_loss < best_val_metric - 1e-6
                current  = val_loss

            if improved:
                best_val_metric = current
                best_state      = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                bad_epochs      = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    print(f"\n    [Early Stop] No improvement for {self.patience} epochs at epoch {epoch}.")
                    break

        label = "Best Val BA" if self.early_stop_on == "val_ba" else "Best Val LogLoss"
        print(f"\n    [{label}]: {best_val_metric:.5f}")
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    def predict_proba(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        return _predict_nodes(
            self.model_, seed_nodes, neighbors, x_num_cpu, x_cat_cpu,
            y_cpu, self.fanouts, self.infer_batch, self.device
        )

    def predict(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        return np.argmax(self.predict_proba(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu), axis=1)


# ============================================================
# SECTION 7 — LOAD DATA & BUILD GRAPH
# ============================================================

print("\n" + "="*60)
print("Loading data...")
print("="*60)

TRAIN_PARQUET = "/content/drive/MyDrive/irrigation_need_v15/train_engineered_v21.parquet"
TEST_PARQUET  = "/content/drive/MyDrive/irrigation_need_v15/test_engineered_v21.parquet"

train_raw = pd.read_parquet(TRAIN_PARQUET)
test_raw  = pd.read_parquet(TEST_PARQUET)
print(f"Raw train: {train_raw.shape} | Raw test: {test_raw.shape}")

# Preprocess
train_pre, test_pre, y_train = preprocess(train_raw, test_raw)

# Feature engineering — passes y_train for decision stump fitting
train_fe, test_fe, stumps = engineer_features(train_pre, test_pre, y_train)

# Encode categoricals (now includes __stump and __digit columns)
Xc_train, Xc_test, cat_cardinalities = encode_categoricals(train_fe, test_fe)

# IMPROVEMENT 2: Rank Percentile Encoding for numerics
Xn_train_raw, Xn_test_raw, rpe = scale_numerics(train_fe, test_fe)

# IMPROVEMENT 4: Autoencoder bottleneck features
ae_train, ae_test = train_autoencoder(Xn_train_raw, Xn_test_raw)

# Append AE features to numeric matrices
Xn_train = np.concatenate([Xn_train_raw, ae_train], axis=1).astype(np.float32)
Xn_test  = np.concatenate([Xn_test_raw,  ae_test],  axis=1).astype(np.float32)
print(f"[Node features] Final numeric dim: {Xn_train.shape[1]} (raw={Xn_train_raw.shape[1]} + AE={AE_LATENT_DIM})")

# Build KNN graph
neighbors = build_knn_graph(train_fe, test_fe, k=K)

n_train = len(train_fe)
n_test  = len(test_fe)
n_all   = n_train + n_test

# Shared tensors (train + test combined)
Xn_all   = np.vstack([Xn_train, Xn_test])
Xc_all   = np.vstack([Xc_train, Xc_test])
y_all_np = np.concatenate([y_train, np.full(n_test, -1, dtype=np.int64)])

x_num_cpu = torch.tensor(Xn_all, dtype=torch.float32).pin_memory()
x_cat_cpu = torch.tensor(Xc_all, dtype=torch.long).pin_memory()
y_cpu     = torch.tensor(y_all_np, dtype=torch.long).pin_memory()

print(f"\n[Tensors] x_num: {tuple(x_num_cpu.shape)} | x_cat: {tuple(x_cat_cpu.shape)}")

classes       = np.arange(N_CLASSES)
cw_values     = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = torch.tensor(cw_values, dtype=torch.float32)
print(f"\n[Class Weights] { {LABEL_INV[i]: round(float(w), 4) for i, w in enumerate(cw_values)} }")


# ============================================================
# SECTION 8 — CROSS-VALIDATION
# ============================================================

print("\n" + "="*60)
print("Starting 5-Fold Cross-Validation")
print("="*60)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs  = np.zeros((n_train, N_CLASSES), dtype=np.float32)
pred_probs = np.zeros((n_test,  N_CLASSES), dtype=np.float32)
test_nodes = np.arange(n_train, n_all, dtype=np.int32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), y_train), 1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold} / {N_FOLDS}")
    print(f"{'='*60}\n")
    t_fold = time.time()

    clf = IrrigationGNNClassifier(
        early_stop_on="val_ba"   # IMPROVEMENT 5: stop on balanced accuracy
    )
    clf.fit(
        train_idx    = tr_idx,
        val_idx      = va_idx,
        y_all        = y_all_np,
        neighbors    = neighbors,
        x_num_cpu    = x_num_cpu,
        x_cat_cpu    = x_cat_cpu,
        y_cpu        = y_cpu,
        cardinalities= cat_cardinalities,
        class_weights= class_weights,
    )

    oof_probs[va_idx] = clf.predict_proba(
        va_idx, neighbors, x_num_cpu, x_cat_cpu, y_cpu
    )
    pred_probs += clf.predict_proba(
        test_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu
    ) / N_FOLDS

    fold_ba      = balanced_accuracy_score(y_train[va_idx], np.argmax(oof_probs[va_idx], axis=1))
    fold_logloss = log_loss(y_train[va_idx], oof_probs[va_idx], labels=[0, 1, 2])
    print(f"\n  [Fold {fold}] BA: {fold_ba:.5f} | LogLoss: {fold_logloss:.5f} | Time: {time.time()-t_fold:.1f}s")

    del clf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# SECTION 9 — OOF EVALUATION
# ============================================================

oof_preds  = np.argmax(oof_probs, axis=1)
cv_bal_acc = balanced_accuracy_score(y_train, oof_preds)
cv_logloss = log_loss(y_train, oof_probs, labels=[0, 1, 2])

print("\n" + "="*60)
print("  OOF EVALUATION")
print("="*60)
print(f"  CV Balanced Accuracy : {cv_bal_acc:.5f}")
print(f"  CV Log-Loss          : {cv_logloss:.5f}")

# Threshold optimisation
best_thresholds  = optimize_thresholds(oof_probs, y_train)
oof_preds_thresh = apply_thresholds(oof_probs, best_thresholds)
ba_thresh        = balanced_accuracy_score(y_train, oof_preds_thresh)
print(f"  Threshold BA         : {ba_thresh:.5f}  (gain: {ba_thresh - cv_bal_acc:+.5f})")

# Genetic bias tuning
best_bias        = genetic_optimize(oof_probs, y_train)
oof_probs_tuned  = apply_bias(oof_probs, best_bias)
oof_preds_tuned  = np.argmax(oof_probs_tuned, axis=1)
ba_tuned         = balanced_accuracy_score(y_train, oof_preds_tuned)
print(f"  Genetic-tuned BA     : {ba_tuned:.5f}  (gain: {ba_tuned - cv_bal_acc:+.5f})")

# Save tuned test predictions
final_test_probs     = apply_bias(pred_probs, best_bias)
test_preds_thresh    = apply_thresholds(pred_probs, best_thresholds)

np.save(f"{OUT_DIR}/thresholds_{VERSION_NB}.npy",       best_thresholds)
np.save(f"{OUT_DIR}/test_preds_thresh_{VERSION_NB}.npy", test_preds_thresh)
np.save(f"{OUT_DIR}/test_preds_tuned_{VERSION_NB}.npy",  final_test_probs)
np.save(f"{OUT_DIR}/oof_tuned_{VERSION_NB}.npy",         oof_probs_tuned)
np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy",               oof_probs)
np.save(f"{OUT_DIR}/test_preds_{VERSION_NB}.npy",        pred_probs)
np.save(f"{OUT_DIR}/y_train_{VERSION_NB}.npy",           y_train)


# ============================================================
# SECTION 10 — SUBMISSION
# ============================================================

print("\n[Submission] Generating predictions...")

test_preds  = np.argmax(pred_probs, axis=1)
test_labels = [LABEL_INV[p] for p in test_preds]

submission = pd.DataFrame({
    "id"   : test_raw["id"],
    TARGET : test_labels,
})
submission.to_csv(f"submission_gnn_{VERSION_NB}.csv", index=False)
print(f"Saved: submission_gnn_{VERSION_NB}.csv")
print(train_raw[TARGET].value_counts())
print("Unique labels in y_train:", np.unique(y_train))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
🚀 Using RAPIDS cuML KNN - Very Fast!
DEVICE: cuda

Loading data...
Raw train: (640000, 111) | Raw test: (270000, 110)
Unique labels: [0 1 2]

[Feature Engineering] Building features...
[Feature Engineering] Fitting decision stumps...
  Stump thresholds: {'Soil_pH': 6.455, 'Soil_Moisture': 24.995, 'Organic_Carbon': 0.415, 'Electrical_Conductivity': 0.755, 'Temperature_C': 30.205, 'Humidity': 90.99, 'Rainfall_mm': 299.79, 'Sunlight_Hours': 10.395, 'Wind_Speed_kmh': 9.995, 'Field_Area_hectare': 7.545, 'Previous_Irrigation_mm': 17.975}
[Feature Engineering] Extracting digit-level features...
[Feature Engineering] Categorical features: 52
[Feature Engineering] Numeric features    : 11 raw + 4 AE latent

[Encode] Encoding categorical node features...
[Encode] Categorical matrix — train: (640000, 52) 

In [6]:
print("Unmapped values:", y_raw[y.isna()].unique())

NameError: name 'y_raw' is not defined

In [10]:
np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy", oof_probs)
np.save(f"{OUT_DIR}/test_preds_{VERSION_NB}.npy", pred_probs)
np.save(f"{OUT_DIR}/y_train_{VERSION_NB}.npy", y_train)

In [9]:
print(train_raw[TARGET].value_counts())

Irrigation_Need
0    375781
1    242874
2     21345
Name: count, dtype: int64


In [8]:
print("Unique labels in y_train:", np.unique(y_train))

Unique labels in y_train: [-9223372036854775808]
